# ClaimIQ – Power BI Gold Export
## Week 8 | `notebooks/06_powerbi_export.ipynb`

Controlled hand-off of validated ClaimIQ Gold data to Power BI.

**Rules**
- Gold objects only.
- Export each Gold table independently.
- No raw/Silver/Trusted Silver detail/Quarantine source.
- No manual correction of exported values.
- No joining independent Gold summary tables merely for convenience.
- Record source, grain, row count, date coverage and read-back checks.


## 1. Gold source register

| Gold table | Grain | Power BI purpose |
|---|---|---|
| `gold_claims_product_monthly_summary` | product × submission month | Claims/product KPIs and trends |
| `gold_claims_payment_monthly` | payment month × payment method | Payment KPIs and trends |
| `gold_claims_region_risk_monthly` | region × risk band × submission month | Regional/risk monitoring |
| `gold_claims_sla_performance_monthly` | product × settlement month | Settlement/SLA performance |

The Region Risk table is based on claims successfully matched to policyholders, so its claim count is not the overall 499-claim population.


In [ ]:
# Configuration
# If your Databricks workspace requires another writable location, change only EXPORT_ROOT.

EXPORT_ROOT = "dbfs:/FileStore/claimiq_powerbi_gold_exports"

GOLD_TABLES = {
    "gold_claims_product_monthly_summary": {
        "order_by": ["submission_month", "product_id"],
        "date_column": "submission_month"
    },
    "gold_claims_payment_monthly": {
        "order_by": ["payment_month", "payment_method"],
        "date_column": "payment_month"
    },
    "gold_claims_region_risk_monthly": {
        "order_by": ["submission_month", "region_code", "risk_band"],
        "date_column": "submission_month"
    },
    "gold_claims_sla_performance_monthly": {
        "order_by": ["settlement_month", "product_id"],
        "date_column": "settlement_month"
    }
}

print("Export root:", EXPORT_ROOT)
print("Gold tables:", list(GOLD_TABLES))


## 2. Gold hand-off checks

Verify every selected Gold table and record row count, column count and date coverage.

In [ ]:
from pyspark.sql import functions as F

profile_rows = []

for table_name, cfg in GOLD_TABLES.items():
    df = spark.table(table_name)
    stats = df.select(
        F.count("*").alias("row_count"),
        F.min(F.col(cfg["date_column"])).alias("min_month"),
        F.max(F.col(cfg["date_column"])).alias("max_month")
    ).collect()[0]

    profile_rows.append((
        table_name,
        int(stats["row_count"]),
        str(stats["min_month"]) if stats["min_month"] is not None else None,
        str(stats["max_month"]) if stats["max_month"] is not None else None,
        len(df.columns)
    ))

profile_df = spark.createDataFrame(
    profile_rows,
    ["gold_table", "row_count", "min_month", "max_month", "column_count"]
)

display(profile_df.orderBy("gold_table"))


## 3. Deterministic Gold exports

Each approved Gold table is exported independently as a small CSV snapshot.

In [ ]:
export_results = []

for table_name, cfg in GOLD_TABLES.items():
    df = spark.table(table_name).orderBy(*cfg["order_by"])
    output_path = f"{EXPORT_ROOT}/{table_name}"

    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", "true")
          .option("emptyValue", "")
          .csv(output_path)
    )

    export_results.append((table_name, output_path, df.count()))

export_results_df = spark.createDataFrame(
    export_results,
    ["gold_table", "export_path", "export_row_count"]
)

display(export_results_df.orderBy("gold_table"))


## 4. Read-back proof

**Pass condition:** exported row count equals the owning Gold row count for every table.

In [ ]:
readback_results = []

for table_name in GOLD_TABLES:
    source_count = spark.table(table_name).count()
    export_path = f"{EXPORT_ROOT}/{table_name}"

    readback_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(export_path)
    )
    readback_count = readback_df.count()

    readback_results.append((
        table_name,
        source_count,
        readback_count,
        "PASS" if source_count == readback_count else "FAIL"
    ))

readback_df = spark.createDataFrame(
    readback_results,
    ["gold_table", "source_row_count", "export_row_count", "row_count_check"]
)

display(readback_df.orderBy("gold_table"))

assert all(r["row_count_check"] == "PASS" for r in readback_df.collect()),     "Gold export row-count reconciliation failed."


## 5. Column/schema proof

Confirm exported column names match the owning Gold table.

In [ ]:
schema_results = []

for table_name in GOLD_TABLES:
    source_cols = spark.table(table_name).columns
    export_path = f"{EXPORT_ROOT}/{table_name}"

    readback_cols = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(export_path)
        .columns
    )

    schema_results.append((
        table_name,
        len(source_cols),
        len(readback_cols),
        source_cols == readback_cols
    ))

schema_df = spark.createDataFrame(
    schema_results,
    ["gold_table", "source_column_count", "export_column_count", "column_order_match"]
)

display(schema_df.orderBy("gold_table"))

assert all(r["column_order_match"] for r in schema_df.collect()),     "Gold export column/schema reconciliation failed."


## 6. Business reconciliation

Expected validated values from the current Gold snapshot:

**Product Gold**
- Total Claims = 499
- Settled Claims = 157
- Total Requested Amount = 120,721,170.51
- Total Approved Amount = 56,409,955.71

**Payment Gold**
- Total Payments = 55
- Distinct Claims Paid = 49
- Gross Paid Amount = 5,519,005.09

**SLA Gold**
- Settled Claims = 156
- Within SLA = 63
- Average settlement days ≈ 20.232

If Gold is regenerated and values intentionally change, update the documented expected values from the new validated Gold output. Never manually edit exported CSV values.


In [ ]:
# Claims/product reconciliation used by the main Power BI page.
product_reconciliation = spark.sql(
    """
    SELECT
        SUM(claim_count) AS total_claims,
        SUM(settled_claim_count) AS total_settled_claims,
        SUM(total_requested_amount) AS total_requested_amount,
        SUM(total_approved_amount) AS total_approved_amount
    FROM gold_claims_product_monthly_summary
    """
)
display(product_reconciliation)

# Payment reconciliation used by the payment page.
payment_reconciliation = spark.sql(
    """
    SELECT
        SUM(payment_count) AS total_payments,
        SUM(distinct_claims_paid) AS total_distinct_claims_paid,
        SUM(gross_paid_amount) AS total_gross_paid,
        SUM(reversal_count) AS total_reversals,
        SUM(reversed_amount) AS total_reversed_amount,
        SUM(net_paid_amount) AS total_net_paid
    FROM gold_claims_payment_monthly
    """
)
display(payment_reconciliation)


## 7. Power BI hand-off checklist

- [ ] Gold tables exist and are validated.
- [ ] Source row counts and date ranges recorded.
- [ ] Each selected Gold table exported independently.
- [ ] Read-back row counts PASS.
- [ ] Column/schema check PASS.
- [ ] Power BI uses Gold exports only.
- [ ] At least three Page 1 values reconcile to Gold.
- [ ] PBIX saved as `dashboard/powerbi_dashboard.pbix`.
- [ ] `dashboard/README.md` updated.
- [ ] `screenshots/week08_*.png` captured.
- [ ] `weekly_logs/week08_log.md` completed, including AI Transparency Note.

### Current report mapping
- Claims & Product Overview → `gold_claims_product_monthly_summary`
- Payment & Risk Analysis → `gold_claims_payment_monthly`
- Region Risk Gold → optional risk visuals
- SLA Gold → optional SLA visuals
